# Chapter 13 &mdash; The Halting Decider $H$, and the Grader's Dilemma

**Concept 2 of the Chapter 13 decomposition:** *The Halting Decider $H$, and the Grader's Dilemma*

A TM can encode any procedure, so it is tempting to want one that decides halting.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter13/Concept-The-Halting-Decider/Concept-The-Halting-Decider.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_PDA        import *
from jove.Def_TM         import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


Since a TM can encode **any** procedure, why not one that takes another machine's
description and its input and answers "does it halt?"

The motivating story is the **grader's dilemma**: you are marking a hundred student
programs. Some loop forever. You would very much like a tool that reads a program and
tells you whether to wait.

Write it as a language:
$$H = \{\langle M, w\rangle : M \text{ halts on } w\}.$$

The intuition that $H$ ought to be decidable is strong and it is **wrong**
(Chapter 14). But the intuition is worth taking seriously first &mdash; you cannot
appreciate the proof without having wanted the tool.

## 2. Definitions

### A machine that halts on some inputs and not others

In [ ]:
Picky = md2mc('''TM
!! halts (in F) iff the tape STARTS with a 1; otherwise it runs right forever
I : 1 ; 1 , R -> F      !! first symbol is 1 -- halt and accept
I : 0 ; 0 , R -> L      !! anything else -- fall into the loop state
I : . ; . , R -> L
L : 0 ; 0 , R -> L
L : 1 ; 1 , R -> L
L : . ; . , R -> L
''')

# --- thin wrappers over Jove's TM runner --------------------------------
# run_tm(T, tape, fuel) returns (truncated-paths, haltList).  A TM HALTS
# when no transition applies, and ACCEPTS if it halts in a final state.
# So an accepting state must have NO outgoing transitions, or the machine
# will run on past it.
def tm_accepts(T, tape, fuel=200):
    trunc, halts = run_tm(T, tape if tape != '' else '.', fuel, chatty=False)
    return any(cfg[0] in T["F"] for cfg, _ in halts)

def tm_halts(T, tape, fuel=200):
    trunc, halts = run_tm(T, tape if tape != '' else '.', fuel, chatty=False)
    return len(halts) > 0

def tm_tape(T, tape, fuel=200):
    trunc, halts = run_tm(T, tape if tape != '' else '.', fuel, chatty=False)
    return [cfg[2].rstrip('.') for cfg, _ in halts]

### A finite-fuel 'decider' &mdash; the obvious wrong idea

In [ ]:
def halts_within(T, tape, fuel):
    trunc, halts = run_tm(T, tape if tape else '.', fuel, chatty=False)
    return len(halts) > 0

## 3. Tests

Some inputs halt, some do not.

In [ ]:
for t in ['1', '11', '01', '001']:
    print("  %-6r halts within 50 steps? %s" % (t, halts_within(Picky, t, 50)))
assert halts_within(Picky, '1', 50)
assert not halts_within(Picky, '001', 50)

**The grader's dilemma, in miniature.** You cannot tell 'slow' from 'never'.

In [ ]:
for fuel in [5, 10, 30, 100]:
    print("  fuel %3d : '0001' halts? %s" % (fuel, halts_within(Picky, '0001', fuel)))
print("\nEvery answer is 'not yet'.  Is it looping, or just slow?")

A machine that is merely **slow** looks identical at small fuel.

In [ ]:
Slow = md2mc('''TM
I : 0 ; 0 , R -> I
I : . ; . , S -> F
''')
for fuel in [2, 5, 20]:
    print("  fuel %2d : Slow on '0000000000' halts? %-6s  Picky on '0001' halts? %s"
          % (fuel, halts_within(Slow, '0' * 10, fuel), halts_within(Picky, '0001', fuel)))
assert halts_within(Slow, '0' * 10, 20)
assert not halts_within(Picky, '0001', 200)
print("\nAt fuel 5 the two are indistinguishable.  One halts later; one never does.")

So define the language and ask whether it is decidable.

In [ ]:
print("H = { <M, w> : M halts on w }")
print()
print("A DECIDER for H would have to answer, for every pair, in finite time.")
print("Running M is a SEMI-decider: it says 'yes' eventually when the answer")
print("is yes, and says nothing at all when the answer is no.")
print()
print("Chapter 14 shows no decider exists.  It is not that we have not found one.")

## 4. Exercises


1. Why is "run it and see" not a decision procedure?
2. Could a decider exist for a *restricted* class of programs? Which?
3. What would you do as the grader, given that no tool exists?

In [ ]:
# Your work for the exercises above.